# HU-EEQI v6 · Notebook 01 — Calibração e justificativa dos pesos

**v6 — calibração depois:** rode **depois do notebook 00** (pilha bruta gravada em `RAW_COLLECTION`) e antes do 02. O estágio A **não calcula mais** Landsat/ERA5/MAIAC: lê a pilha bruta nos mesmos pontos fixos de antes, numa **única task** (`calib_v6_bruto.csv` no Drive). Os estágios B e C não mudaram.

**Rode uma vez, antes do notebook 02.** A v6 muda os descritores brutos (harmonização Landsat, nova máscara QA, HEAT com temperatura do ar, AQ com QA), então os limites e pesos do `calibration2.json` **não valem mais**.

O que este notebook faz:

| Etapa | O que muda em relação à v3 |
|---|---|
| A. Amostragem | Todos os anos (2000–2025), 30 m, só dentro do domínio fixo. **v5 (rápida):** painel fixo de `N_POINTS_PER_CITY` pixels por capital, sorteado **uma vez** só na máscara do domínio; a pilha de descritores é calculada **apenas nesses pontos** (`sampleRegions`), e o HEAT uma vez por célula ERA5-Land. Uma task por capital e ano, em paralelo. Antes (v4): `stratifiedSample` sobre a pilha inteira, que calculava todos os pixels de 30 m do município (inclusive fora da área urbana); com o HEAT pixel a pixel, passava de 3 horas por ano. |
| B. Limites | Percentis agrupados (padrão p1–p99), mais p2–p98 e p5–p95 guardados para a análise de sensibilidade. |
| B. Diagnóstico | % de pixels saturados por ano e descritor, medianas por ano e por sensor (degrau 2012/2013), correlação e VIF. |
| B. Pesos [7] | CRITIC (corrigido: antes as colunas eram truncadas uma a uma e as linhas se desalinhavam), entropia (EWM), PCA (PC1, como no RSEI) e pesos iguais. IC de 95% do CRITIC por bootstrap em blocos de cidade, e CRITIC ano a ano (estabilidade temporal). |
| C. Saída | `calibration_v6.json` com limites, todos os conjuntos de pesos e um retrato da configuração (o notebook 02 recusa rodar se a configuração mudou). |

### Por que a amostragem v5 é rápida e continua defensável
- **O gargalo da v4:** `stratifiedSample` precisa da pilha em todos os pixels da região para sortear entre os válidos. Nos municípios grandes (Porto Velho ~34 mil km², Manaus ~11 mil km², Boa Vista ~5,7 mil km²), isso calcula dezenas de milhões de pixels de 30 m, quase todos fora do domínio urbano. Em cada um, o HEAT juntava 5 horas × 365 dias do ERA5 e ordenava 365 valores.
- **v5:** os pixels são sorteados só na máscara do domínio (uma operação barata), e os descritores são calculados apenas nesses pontos. O HEAT é calculado uma vez por célula de 0,1° do ERA5-Land, com o **mesmo valor** de antes (diferença 0 no teste).
- **Teste no modo interativo (1 capital × 1 ano, 400 pontos):** em Porto Velho, a v5 levou 150 s e a v4 estourou o limite de 5 min sem terminar. Em Aracaju (pequena), as duas ficaram entre 1,5 e 5 min.
- **Painel fixo:** os mesmos pixels em todos os anos. O diagnóstico temporal (saturação e medianas por ano) compara os mesmos locais e não mistura variação de amostra com variação do tempo.
- **Pixel-ano sem algum descritor** (nuvem etc.) sai da amostra, como na v4 (`dropNulls`).
- **Peso igual por capital e ano:** o estágio B fica com até `CALIB_BALANCE_N` amostras válidas por capital e ano (as de menor `prand`, uma ordem aleatória fixa dos pontos), o mesmo desenho da v4. A escolha é a mesma no script do GEE.

In [ ]:
# @title Configuração HU-EEQI v6  (mantenha IDÊNTICA em todos os notebooks)
import os
# ---- Projeto, conta e assets --------------------------------------------------
# Edite os valores aqui. Se existir uma variável de ambiente (ou um Secret do Colab,
# ícone de chave) com o nome indicado no comentário, ela VENCE o valor digitado.
EE_PROJECT        = ""                                  # @param {type:"string"}
EE_ACCOUNT        = ""                  # @param {type:"string"}
CITIES_ASSET      = "projects/rnvuefsppgm/assets/Pol_capitais"        # @param {type:"string"}
CITY_NAME_PROP    = "NM_MUN"                                          # @param {type:"string"}
# Destinos: projeto Cloud mapbiomas-caatinga-cloud04 (formato projects/<projeto>/assets/<nome>)
RAW_COLLECTION    = "projects/mapbiomas-caatinga-cloud04/assets/HU-EEQI_v6_raw"      # @param {type:"string"}
OUT_COLLECTION    = "projects/mapbiomas-caatinga-cloud04/assets/HU-EEQI_v6"          # @param {type:"string"}
FILLED_COLLECTION = "projects/mapbiomas-caatinga-cloud04/assets/HU-EEQI_v6_filled"   # @param {type:"string"}
TRENDS_COLLECTION = "projects/mapbiomas-caatinga-cloud04/assets/TRENDS_HU-EEQI_v6"   # @param {type:"string"}
GDP_ASSET         = "projects/sat-io/open-datasets/GRIDDED_HDI_GDP/adm1_gdp_perCapita_1990_2022"   # @param {type:"string"}
MAPBIOMAS_ASSET   = "projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1"   # @param {type:"string"}
CENSUS_2010_ASSET = ""   # @param {type:"string"}
CENSUS_2022_ASSET = ""   # @param {type:"string"}
# variável de ambiente / Secret -> variável acima
_ENV_NAMES = {"EE_PROJECT": "EE_PROJECT", "HU_EE_ACCOUNT": "EE_ACCOUNT",
              "HU_CITIES_ASSET": "CITIES_ASSET", "HU_EEQI_V6_COLLECTION": "OUT_COLLECTION",
              "HU_EEQI_V6_RAW_COLLECTION": "RAW_COLLECTION",
              "HU_EEQI_V6_FILLED_COLLECTION": "FILLED_COLLECTION",
              "HU_EEQI_V6_TRENDS_COLLECTION": "TRENDS_COLLECTION", "HU_GDP_ASSET": "GDP_ASSET",
              "HU_MAPBIOMAS_ASSET": "MAPBIOMAS_ASSET",
              "HU_CENSUS_2010_ASSET": "CENSUS_2010_ASSET", "HU_CENSUS_2022_ASSET": "CENSUS_2022_ASSET"}

def _from_env_or_secret(env_name):
    v = os.environ.get(env_name)
    if v:
        return v
    try:
        from google.colab import userdata
        return userdata.get(env_name) or ""
    except Exception:          # fora do Colab, sem Secret ou sem acesso ao Secret
        return ""

for _env, _var in _ENV_NAMES.items():
    _v = _from_env_or_secret(_env)
    if _v:
        globals()[_var] = _v.strip()
for _var in ["EE_PROJECT", "CITIES_ASSET", "OUT_COLLECTION"]:
    if not globals()[_var]:
        raise ValueError(f"Preencha {_var} na célula de configuração.")
print("Projeto :", EE_PROJECT, "| conta:", EE_ACCOUNT)
print("Capitais:", CITIES_ASSET, "| pilha bruta:", RAW_COLLECTION, "| índice:", OUT_COLLECTION)
print("Série preenchida:", FILLED_COLLECTION, "| tendências:", TRENDS_COLLECTION)
# assets que a célula de autenticação testa (leitura) antes de qualquer task
READ_ASSETS = {"capitais": CITIES_ASSET, "PIB (V_SOURCE='gdp_adm1')": GDP_ASSET,
               "MapBiomas (notebooks 03/05)": MAPBIOMAS_ASSET}

# ---- saídas no Drive -----------------------------------------------------------
DRIVE_FOLDER    = "hu_eeqi_v6"            # pasta na raiz do Meu Drive (tabelas e JSON)
CALIB_JSON_NAME = "calibration_v6.json"   # DRIVE_DIR e CALIB_JSON são definidos na autenticação
OLD_CALIB_REL   = "hu_eeqi/calibration2.json"   # calibração v3 (só para comparar), relativo ao Meu Drive

# ---- série temporal ----------------------------------------------------------
YEAR_START, YEAR_END = 2000, 2025
YEARS = list(range(YEAR_START, YEAR_END + 1))

# ---- escalas -----------------------------------------------------------------
# Cada camada fica na resolução ORIGINAL (Landsat 30 m, ERA5-Land ~9 km, MAIAC 1 km,
# GHS-POP 100 m, PIB adm1): nenhuma reprojeção, agregação ou interpolação. O índice é
# calculado pixel a pixel na grade de exportação (30 m, a do Landsat).
EQ_SCALE       = 30       # escala de exportação = resolução do Landsat (m)
EXPORT_CRS     = "EPSG:4326"   # mesmo CRS de exportação da v3
AQ_STATS_SCALE = 1000     # só p/ a estatística escalar da cidade (mediana de AQ)
V_STATS_SCALE  = 1000     # só p/ os percentis p2/p98 de V na cidade

# ---- domínio de análise (FIXO no tempo) ---------------------------------------
APPLY_URBAN_MASK = True   # True = união GHS-SMOD >= URBAN_SMOD_MIN em 2000-2025
URBAN_SMOD_MIN   = 21     # 21 suburbano, 22 semi-denso, 23 denso, 30 centro urbano
WATER_OCC_MAX    = 50     # % de ocorrência JRC-GSW acima da qual o pixel é água

# ---- [1] harmonização TM/ETM+ -> OLI (Roy et al., 2016) -----------------------
HARMONIZE  = True
ROY_METHOD = "OLS"        # "OLS" ou "RMA" (Tabela 2 de Roy et al., 2016)
L7_POLICY  = "no_oli_years"   # "no_oli_years": ETM+ só até 2012 | "always"

# ---- [2] qualidade Landsat -----------------------------------------------------
ST_QA_MAX_K      = None   # incerteza máxima da LST (K); None = sem limite. v6: com 2 K nenhum pixel
                         # passava (ST_QA fica em 2,5–7 K nas capitais); a LST é só diagnóstico
MIN_OBS          = 3      # mínimo de observações limpas no ano por pixel
COMPOSITE_MONTHS = None   # ex.: (6, 9) = jun-set (estação seca); None = ano inteiro
CITY_MONTHS      = {}     # janela por cidade, ex.: {"Manaus": (7, 10)}

# ---- [4]/[8] HEAT: temperatura do ar e ponto de orvalho (ERA5-Land horário) ---
HEAT_METHOD      = "nws_hi"           # "nws_hi" (Rothfusz/NWS) | "steadman_at"
HEAT_LOCAL_HOURS = (12, 16)           # janela da tarde, em hora LOCAL
HEAT_STAT        = "median_daily_max" # mediana anual do máximo diário | "p95_daily_max"
HEAT_XC_THRESH_C = 32.2               # limiar NWS "extreme caution" (90 °F)
# HEAT calculado 1 vez por célula do ERA5-Land (0,1°) e não 1 vez por pixel de 30 m.
# Mesmo valor (testado: diferença 0): cada pixel de 30 m já recebia o valor da célula
# em que cai. Sem reamostragem nem agregação; só evita repetir a mesma conta ~90 mil vezes.
NATIVE_GRID_EVAL = True
# v6: célula do ERA5-Land sem dado (o ERA5-Land mascara o mar; ~7% de Aracaju) recebe a média das
# células vizinhas imediatas com dado (3×3, grade nativa, mesmo ano); marcada na banda HEAT_filled.
HEAT_FILL_NEIGHBORS = True

# ---- AQ ------------------------------------------------------------------------
AQ_SOURCE          = "maiac"  # "maiac" (AOD 550 nm) | "pm25" (coleção abaixo)
AQ_REQUIRE_BEST_QA = True     # MAIAC: céu claro + sem adjacência + QA de AOD "best"
AOD_MIN_OBS        = 10       # mínimo de retrievals válidos no ano por pixel
AQ_FILL            = "city_median"  # lacuna -> mediana do ano na cidade | "none" (fica sem dado)
PM25_COLLECTION    = None     # ex.: ImageCollection anual de PM2.5 (se tiver acesso)
PM25_BAND          = None

# ---- [3] população -------------------------------------------------------------
POP_SOURCE = "ghs"                        # "ghs" (GHS-POP R2023A) | "worldpop"
GHS_POP_IC = "JRC/GHSL/P2023A/GHS_POP"
POP_W_EPS  = 1.0     # hab por célula de 100 m somados ao peso: área vazia -> peso ~uniforme

# ---- vulnerabilidade / justiça -------------------------------------------------
USE_JUSTICE     = True
V_SOURCE        = "gdp_adm1"   # "census" (RECOMENDADO) | "gdp_adm1" (GDP_ASSET acima)
GDP_YEAR_START, GDP_YEAR_END = 1990, 2022
CENSUS_INCOME_PROP = ""   # nome da propriedade de renda nos assets de setores censitários
V_CENSUS = {y: {"asset": a, "prop": CENSUS_INCOME_PROP}
            for y, a in ((2010, CENSUS_2010_ASSET), (2022, CENSUS_2022_ASSET)) if a}
V_MIN_REL_RANGE = 1e-3          # (p98-p2)/|p98| mínimo p/ J ser informativo
J_FALLBACK      = "renormalize" # J degenerado: "renormalize" | "to_eq"

# ---- fusão  HU-EEQI = EQ^a · E^b · J^c ----------------------------------------
ALPHA, BETA, GAMMA = 0.6, 0.2, 0.2
WIN_RADIUS_M = 2000
NI_FLOOR     = 0.01

# ---- [7] calibração --------------------------------------------------------------
BOUNDS_PCT    = (1, 99)     # limites agrupados (todas as cidades e todos os anos)
WEIGHT_METHOD = "critic"    # "critic" | "ewm" | "pca" | "equal"

DESCRIPTORS = ["NDVI", "Wet", "NDBSI", "HEAT", "AQ"]
DIRECTION   = {"NDVI": "benefit", "Wet": "benefit", "NDBSI": "cost",
               "HEAT": "cost", "AQ": "cost"}

# ---- [6] exportação --------------------------------------------------------------
EXPORT_DIAGNOSTICS = True   # descritores brutos + QA (obrigatório p/ os notebooks 03 e 05)
SKIP_EXISTING      = True   # pula assets já exportados (permite retomar)
VERSION = "v6"

# ---- painel fixo de pixels (notebooks 03 e 05) -----------------------------------
N_PANEL_PER_CITY = 400      # pixels por capital (amostra estratificada por unidade)
PANEL_SEED       = 7        # semente base; capital i usa PANEL_SEED + i
MB_YEARS         = (2000, 2023)   # anos MapBiomas usados para definir áreas estáveis

# ---- tendências (notebooks 04 e 05) ------------------------------------------------
ALPHA_SIG     = 0.05        # nível de significância (o mesmo p < 0,05 do Trends v3)
FDR_Q         = ALPHA_SIG   # nível q de Benjamini–Hochberg (item 2)
FDR_HIST_BINS = 50000       # resolução do histograma de p em [0, ALPHA_SIG] (BH conservador)
R1_THRESHOLD  = 0.3         # limiar de autocorrelação lag-1 pedido na revisão

# ---- [9] preenchimento de lacunas (notebook 02b; usado nos 04 e 05) -----------------
# Lacuna = pixel do domínio sem valor no ano (nuvem/sombra, < MIN_OBS observações limpas, ano inteiro
# sem imagem, HEAT/AQ/LST sem dado). Cascata pixel a pixel e banda a banda, sobre a série anual JÁ
# exportada, em TODAS as bandas contínuas (índice, componentes, descritores, LST, POP e V):
#   fill_type 0 = observado
#   fill_type 1 = interpolação linear no tempo entre o último ano com dado antes e o primeiro
#                 depois (lacuna de 1 ano = média dos dois; vale também para o ano inteiro vazio)
#   fill_type 2 = ponta da série (só um lado tem dado): ano com dado mais próximo
#   fill_type 3 = pixel sem dado em nenhum ano: média dos pixels com valor no mesmo ano numa janela
#                 de WIN_RADIUS_M (a mesma janela de E e J)
#   fill_type 4 = nenhum pixel com valor nessa janela: mediana da capital no ano
# filled_mask: bit i = banda FILL_BANDS[i] preenchida (não observada) naquele pixel-ano.
FILL_BANDS      = ["HU_EEQI", "EQ", "E_local", "J_local",
                   "NI_NDVI", "NI_Wet", "NI_NDBSI", "NI_HEAT", "NI_AQ",
                   "NDVI", "Wet", "NDBSI", "HEAT", "AQ", "LST", "LST_anom", "HEAT_ndays_xc", "POP", "V"]
FILL_KEY_BAND   = "HU_EEQI"   # fill_type e gap_years descrevem esta banda
# contagens e marcas copiadas do ano (0 = sem observação/sem marca naquele ano); urban é recalculada
FILL_COPY_BANDS = ["n_obs", "sensor_flag", "n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "n_aod",
                   "AQ_filled", "HEAT_filled", "NI_clamped"]
FILL_REPORT_GAP = 3           # resumo: pixels preenchidos com lacuna maior que 3 anos
TREND_SOURCE    = "filled"    # 04/05: "filled" = série preenchida (02b) | "observed" = só anos observados


In [ ]:
# @title Autenticar (conta EE_ACCOUNT), iniciar o Earth Engine no projeto e montar o Drive
import json, os, subprocess, sys

def _pip(pkg, module=None):
    """Instala um pacote só se ainda não estiver disponível (Colab ou máquina local)."""
    try:
        __import__(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_pip("earthengine-api", "ee")
import ee
try:
    from google.colab import auth as _colab_auth, drive as _colab_drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def _active_account():
    """Melhor esforço: e-mail da credencial ativa (gcloud; senão, userinfo). '' se não souber."""
    try:
        r = subprocess.run(["gcloud", "auth", "list", "--filter=status:ACTIVE",
                            "--format=value(account)"], capture_output=True, text=True, timeout=60)
        lines = [l.strip() for l in r.stdout.splitlines() if "@" in l]
        if lines:
            return lines[0]
    except Exception:
        pass
    try:
        import google.auth, google.auth.transport.requests, requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        r = requests.get("https://www.googleapis.com/oauth2/v3/userinfo",
                         headers={"Authorization": "Bearer " + creds.token}, timeout=30)
        return r.json().get("email", "")
    except Exception:
        return ""

if IN_COLAB:
    # 1) Pop-up do Colab: escolha a conta EE_ACCOUNT. O Earth Engine usa essa mesma credencial
    #    (ee.Initialize lê as "application default credentials" criadas aqui).
    _colab_auth.authenticate_user()
    # Um arquivo de credenciais do Earth Engine deixado no runtime (ee.Authenticate() anterior,
    # talvez com outra conta) teria prioridade sobre a conta escolhida no pop-up: remove.
    _old_cred = os.path.expanduser("~/.config/earthengine/credentials")
    if os.path.exists(_old_cred):
        os.remove(_old_cred)
        print("Credencial antiga do Earth Engine removida do runtime.")
    ACCOUNT = _active_account()
    if ACCOUNT and EE_ACCOUNT and ACCOUNT.lower() != EE_ACCOUNT.lower():
        raise RuntimeError(f"Conta autenticada = {ACCOUNT}, esperada = {EE_ACCOUNT}. Vá em "
                           "Runtime > Disconnect and delete runtime e autentique com a conta certa "
                           "(ou mude EE_ACCOUNT na configuração).")
    print("Conta autenticada:", ACCOUNT or "(não foi possível confirmar o e-mail)")
else:
    # Máquina local: credenciais do `earthengine authenticate` (ou pop-up na 1ª vez)
    ACCOUNT = ""
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate()
# 2) Projeto: falha AQUI (e não no meio das exportações) se a conta não puder usá-lo
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception as e:
    raise PermissionError(
        f"A conta {ACCOUNT or 'autenticada'} não conseguiu usar o projeto {EE_PROJECT}. "
        "Confira o nome do projeto e peça, nesse projeto, os papéis "
        "'Service Usage Consumer' e 'Earth Engine Resource Writer'.\nDetalhe: " + str(e)[:300])
print(f"Earth Engine OK | projeto {EE_PROJECT} | tasks criadas por {ACCOUNT or 'esta conta'}")
# 3) Leitura dos assets de entrada
for _nome, _aid in READ_ASSETS.items():
    if not _aid:
        continue
    try:
        ee.data.getAsset(_aid)
        print("  leitura OK :", _nome, "->", _aid)
    except Exception as e:
        raise PermissionError(f"Sem acesso de leitura ao asset '{_nome}': {_aid}\n"
                              f"Detalhe: {str(e)[:200]}")
# 4) Drive (Colab) ou pasta local. Monte com a MESMA conta: os exports toDrive vão para ela.
if IN_COLAB:
    _colab_drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive"
else:
    DRIVE_ROOT = os.path.abspath(os.environ.get("HU_DRIVE_ROOT", "."))
DRIVE_DIR      = f"{DRIVE_ROOT}/{DRIVE_FOLDER}"
CALIB_JSON     = f"{DRIVE_DIR}/{globals().get('CALIB_JSON_NAME', 'calibration.json')}"
OLD_CALIB_JSON = f"{DRIVE_ROOT}/{globals().get('OLD_CALIB_REL', 'hu_eeqi/calibration2.json')}"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Pasta de trabalho:", DRIVE_DIR)

In [ ]:
# @title Biblioteca HU-EEQI v6  (não editar — idêntica em todos os notebooks)
import ee, json, math, re, unicodedata

# ============================================================================
# 0) Constantes de dados
# ============================================================================
OPT_BANDS   = ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
SENSOR_CODE = {"TM": 1, "ETM": 2, "OLI": 4, "OLI2": 8}      # bits do sensor_flag
LS_COLLECTIONS = [("LANDSAT/LT05/C02/T1_L2", "TM"),
                  ("LANDSAT/LE07/C02/T1_L2", "ETM"),
                  ("LANDSAT/LC08/C02/T1_L2", "OLI"),
                  ("LANDSAT/LC09/C02/T1_L2", "OLI2")]
SENSOR_YEARS = {"TM": (1984, 2012), "ETM": (1999, 2024),
                "OLI": (2013, 2100), "OLI2": (2021, 2100)}
SR_SRC = {"TM":   ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"],
          "ETM":  ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"],
          "OLI":  ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"],
          "OLI2": ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]}
ST_SRC = {"TM": "ST_B6", "ETM": "ST_B6", "OLI": "ST_B10", "OLI2": "ST_B10"}

# [1] Roy et al. (2016) RSE 185:57-70, Tabela 2 — transformação ETM+ -> OLI,
#     refletância de superfície, ordem BLUE, GREEN, RED, NIR, SWIR1, SWIR2.
#     Aplicada também ao TM (assumido equivalente ao ETM+). Confira na tabela
#     original antes da submissão.
ROY_COEF = {
    "OLS": {"slope": [0.8474, 0.8483, 0.9047, 0.8462, 0.8937, 0.9071],
            "itcp":  [0.0003, 0.0088, 0.0061, 0.0412, 0.0254, 0.0172]},
    "RMA": {"slope": [0.9785, 0.9542, 0.9825, 1.0073, 1.0171, 0.9949],
            "itcp":  [-0.0095, -0.0016, -0.0022, -0.0021, -0.0030, 0.0029]},
}

# Tasseled-cap Wetness (Crist, 1985; refletância). Após a harmonização, todos os
# sensores estão na escala OLI, então UM único conjunto é aplicado à série toda.
TCW = {"BLUE": 0.1509, "GREEN": 0.1973, "RED": 0.3279, "NIR": 0.3406,
       "SWIR1": -0.7112, "SWIR2": -0.4572}

SMOD_IC     = "JRC/GHSL/P2023A/GHS_SMOD_V2-0"
GSW_IMG     = "JRC/GSW1_4/GlobalSurfaceWater"
ERA5_HOURLY = "ECMWF/ERA5_LAND/HOURLY"
MAIAC_IC    = "MODIS/061/MCD19A2_GRANULES"
WORLDPOP_IC = "WorldPop/GP/100m/pop"

CORE_BANDS = ["HU_EEQI", "EQ", "E_local", "J_local",
              "NI_NDVI", "NI_Wet", "NI_NDBSI", "NI_HEAT", "NI_AQ",
              "n_obs", "sensor_flag", "POP", "V", "urban", "domain"]
DIAG_BANDS = ["NDVI", "Wet", "NDBSI", "HEAT", "AQ", "NI_clamped",
              "n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "LST", "LST_anom",
              "n_aod", "AQ_filled", "HEAT_ndays_xc", "HEAT_filled"]


# ============================================================================
# 1) Utilidades, cidades e domínio
# ============================================================================
def ascii_safe(s):
    a = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Za-z0-9.-]", "_", a)[:40] or "city"

def _setter(props):
    def f(feat):
        return feat.set(props)
    return f

def cities():
    return ee.FeatureCollection(CITIES_ASSET)

def city_contexts():
    """Lê nomes e longitudes de uma vez (2 getInfo) e monta o contexto por cidade."""
    fc = cities()
    names = fc.aggregate_array(CITY_NAME_PROP).getInfo()
    lons = (fc.map(lambda f: ee.Feature(None, {
                "lon": f.geometry().centroid(1000).coordinates().get(0)}))
              .aggregate_array("lon").getInfo())
    flist = fc.toList(fc.size())
    ctxs = []
    for i, (name, lon) in enumerate(zip(names, lons)):
        region = ee.Feature(flist.get(i)).geometry()
        ctxs.append({
            "idx": i, "name": name, "safe": ascii_safe(name), "region": region,
            # região com margem: evita efeito de borda nos focais E/J
            "region_buf": region.buffer(WIN_RADIUS_M + 1000, 100),   # margem p/ a janela de E/J
            "utc_offset": int(round(lon / 15.0)),    # hora SOLAR (lon/15): Brasília -3, Rio Branco -5, Recife -2
            "months": CITY_MONTHS.get(name, COMPOSITE_MONTHS),
        })
    return ctxs

def domain_mask():
    """Domínio FIXO: terra (JRC-GSW) ∩ união urbana GHS-SMOD 2000-2025."""
    land = ee.Image(GSW_IMG).select("occurrence").unmask(0).lt(WATER_OCC_MAX)
    if not APPLY_URBAN_MASK:
        return land.rename("domain")
    urb = (ee.ImageCollection(SMOD_IC)
           .filterDate(f"{YEAR_START}-01-01", f"{YEAR_END + 1}-01-01")
           .map(lambda i: i.select(0).gte(URBAN_SMOD_MIN))
           .max())
    return land.And(urb).rename("domain")

def urban_mask(year):
    """Urbano do ano (época SMOD mais próxima) — só informativo; o domínio é fixo."""
    e = min(max(int(5 * round(year / 5.0)), 1975), 2030)
    smod = ee.ImageCollection(SMOD_IC).filter(ee.Filter.calendarRange(e, e, "year")).first()
    return ee.Image(smod).select(0).gte(URBAN_SMOD_MIN).rename("urban")

def config_snapshot():
    """Parâmetros que mudam os descritores brutos: a calibração tem de bater com eles."""
    snap = {"HARMONIZE": HARMONIZE, "ROY_METHOD": ROY_METHOD, "L7_POLICY": L7_POLICY,
            "ST_QA_MAX_K": ST_QA_MAX_K, "MIN_OBS": MIN_OBS,
            "COMPOSITE_MONTHS": COMPOSITE_MONTHS, "CITY_MONTHS": CITY_MONTHS,
            "HEAT_METHOD": HEAT_METHOD, "HEAT_LOCAL_HOURS": HEAT_LOCAL_HOURS,
            "HEAT_STAT": HEAT_STAT, "HEAT_FILL_NEIGHBORS": HEAT_FILL_NEIGHBORS, "AQ_SOURCE": AQ_SOURCE,
            "AQ_REQUIRE_BEST_QA": AQ_REQUIRE_BEST_QA, "AOD_MIN_OBS": AOD_MIN_OBS,
            "AQ_FILL": AQ_FILL, "PM25_COLLECTION": PM25_COLLECTION,
            "APPLY_URBAN_MASK": APPLY_URBAN_MASK, "URBAN_SMOD_MIN": URBAN_SMOD_MIN,
            "WATER_OCC_MAX": WATER_OCC_MAX, "EQ_SCALE": EQ_SCALE,
            "ANNUAL_STAT": "median", "BOUNDS_PCT": BOUNDS_PCT,
            "DESCRIPTORS": DESCRIPTORS, "DIRECTION": DIRECTION}
    return json.loads(json.dumps(snap))

def check_calibration(calib):
    old = calib.get("config", {})
    cur = config_snapshot()
    diff = {k: {"calib": old.get(k), "atual": v} for k, v in cur.items() if old.get(k) != v}
    if diff:
        raise ValueError("Calibração incompatível com a configuração atual — rode o "
                         "notebook 01 de novo.\n" + json.dumps(diff, ensure_ascii=False, indent=1))
    print("Calibração compatível (criada em %s, n=%s amostras)."
          % (calib.get("created"), calib.get("n_samples")))


# ============================================================================
# 2) Landsat C2 L2 — [2] máscara QA, [1] harmonização, composição com n_obs
# ============================================================================
def _prep_landsat(img, sensor):
    qa = img.select("QA_PIXEL")
    # bits 0-4: fill, dilated cloud, cirrus, cloud, cloud shadow -> todos têm de ser 0
    clear = qa.bitwiseAnd(0b11111).eq(0)
    unsat = img.select("QA_RADSAT").eq(0)              # nenhuma banda saturada
    good  = clear.And(unsat)

    sr = img.select(SR_SRC[sensor], OPT_BANDS).multiply(0.0000275).add(-0.2)
    if HARMONIZE and sensor in ("TM", "ETM"):
        c = ROY_COEF[ROY_METHOD]
        sr = (sr.multiply(ee.Image.constant(c["slope"]))
                .add(ee.Image.constant(c["itcp"]))
                .rename(OPT_BANDS))
    # remove refletâncias fisicamente impossíveis (bordas de nuvem, sombra profunda)
    phys = sr.reduce(ee.Reducer.min()).gt(0).And(sr.reduce(ee.Reducer.max()).lt(1))
    sr = sr.updateMask(good.And(phys))

    lst = (img.select(ST_SRC[sensor]).multiply(0.00341802).add(149.0)
              .subtract(273.15).rename("LST"))
    st_unc = img.select("ST_QA").multiply(0.01)       # incerteza em K
    lst = lst.updateMask(good).updateMask(st_unc.gt(0))
    if ST_QA_MAX_K is not None:
        lst = lst.updateMask(st_unc.lte(ST_QA_MAX_K))

    sensor_band = (ee.Image.constant(SENSOR_CODE[sensor]).toInt()
                   .rename("sensor").updateMask(sr.select("NIR").mask()))
    out = sr.addBands(lst).addBands(sensor_band).set("sensor", sensor)
    return ee.Image(out.copyProperties(img, ["system:time_start"]))

def _prep_fn(sensor):
    def f(im):
        return _prep_landsat(im, sensor)
    return f

def _date_filter(col, year, months):
    col = col.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
    if months:
        # calendarRange com início > fim faz a volta do ano (ex.: (11, 2))
        col = col.filter(ee.Filter.calendarRange(int(months[0]), int(months[1]), "month"))
    return col

def landsat_collection(year, region, months):
    cols = []
    for cid, sensor in LS_COLLECTIONS:
        y0, y1 = SENSOR_YEARS[sensor]
        if not (y0 <= year <= y1):
            continue
        if sensor == "ETM" and L7_POLICY == "no_oli_years" and year >= 2013:
            continue     # SLC-off + deriva de órbita: só usa ETM+ quando não há OLI
        c = _date_filter(ee.ImageCollection(cid).filterBounds(region), year, months)
        cols.append(c.map(_prep_fn(sensor)))
    col = cols[0]
    for c in cols[1:]:
        col = col.merge(c)
    return col

def _count_eq(code):
    def f(i):
        return i.eq(code)
    return f

def landsat_composite(year, ctx):
    """Mediana harmonizada + bandas de QA (n_obs, sensores, LST diagnóstica)."""
    col = landsat_collection(year, ctx["region_buf"], ctx["months"])
    n_obs = col.select("NIR").count().rename("n_obs")
    opt = col.select(OPT_BANDS).median().updateMask(n_obs.gte(MIN_OBS))
    sens = col.select("sensor")
    sensor_flag = sens.reduce(ee.Reducer.bitwiseOr()).rename("sensor_flag")
    n_tm  = sens.map(_count_eq(1)).sum().unmask(0).rename("n_TM")
    n_etm = sens.map(_count_eq(2)).sum().unmask(0).rename("n_ETM")
    n_oli = sens.map(lambda i: i.gte(4)).sum().unmask(0).rename("n_OLI")
    lst   = col.select("LST").median().rename("LST")
    n_th  = col.select("LST").count().rename("n_obs_thermal")
    qa = ee.Image.cat([n_obs.unmask(0), sensor_flag, n_tm, n_etm, n_oli, n_th, lst])
    return opt.clip(ctx["region_buf"]), qa.clip(ctx["region_buf"])

def descriptors_optical(comp):
    nir, red = comp.select("NIR"), comp.select("RED")
    swir1, blue, green = comp.select("SWIR1"), comp.select("BLUE"), comp.select("GREEN")
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    wet = comp.select("BLUE").multiply(TCW["BLUE"])
    for b in ["GREEN", "RED", "NIR", "SWIR1", "SWIR2"]:
        wet = wet.add(comp.select(b).multiply(TCW[b]))
    wet = wet.rename("Wet")
    # NDBSI = (IBI + SI) / 2  (Xu, 2008; Xu, 2013)
    t1 = swir1.multiply(2).divide(swir1.add(nir))
    t2 = nir.divide(nir.add(red)).add(green.divide(green.add(swir1)))
    ibi = t1.subtract(t2).divide(t1.add(t2))
    si = swir1.add(red).subtract(nir.add(blue)).divide(swir1.add(red).add(nir.add(blue)))
    ndbsi = ibi.add(si).divide(2).rename("NDBSI")
    return ee.Image.cat([ndvi, wet, ndbsi])


# ============================================================================
# 3) [4]/[8] HEAT — índice de calor com TEMPERATURA DO AR (ERA5-Land horário)
#     Nada de LST aqui. T e Td vêm do mesmo instante; RH via Magnus.
# ============================================================================
def _es_hpa(t):
    """Pressão de saturação (hPa), Magnus — Alduchov & Eskridge (1996); t em °C."""
    return t.expression("6.1094 * exp(17.625 * T / (T + 243.04))", {"T": t})

def heat_index_nws(t_c, rh):
    """Heat index NWS (Rothfusz 1990 + ajustes NWS). Entrada °C e %, saída °C."""
    T = t_c.multiply(1.8).add(32)                     # °F
    simple = T.expression("0.5 * (T + 61.0 + (T - 68.0) * 1.2 + RH * 0.094)",
                          {"T": T, "RH": rh})
    roth = T.expression(
        "-42.379 + 2.04901523*T + 10.14333127*RH - 0.22475541*T*RH"
        " - 0.00683783*T*T - 0.05481717*RH*RH + 0.00122874*T*T*RH"
        " + 0.00085282*T*RH*RH - 0.00000199*T*T*RH*RH", {"T": T, "RH": rh})
    adj_dry = T.expression("((13 - RH) / 4) * sqrt((17 - abs(T - 95)) / 17)",
                           {"T": T, "RH": rh})
    adj_wet = T.expression("((RH - 85) / 10) * ((87 - T) / 5)", {"T": T, "RH": rh})
    roth = roth.where(rh.lt(13).And(T.gte(80)).And(T.lte(112)), roth.subtract(adj_dry))
    roth = roth.where(rh.gt(85).And(T.gte(80)).And(T.lte(87)), roth.add(adj_wet))
    hi_f = simple.where(simple.add(T).divide(2).gte(80), roth)
    return hi_f.subtract(32).divide(1.8)

def _hourly_heat(img):
    t  = img.select("temperature_2m").subtract(273.15)
    td = img.select("dewpoint_temperature_2m").subtract(273.15)
    e  = _es_hpa(td)                                  # pressão de vapor real (hPa)
    rh = e.divide(_es_hpa(t)).multiply(100).clamp(0, 100)
    if HEAT_METHOD == "steadman_at":
        # Steadman (1994), temperatura aparente à sombra: AT = Ta + 0.33e - 0.70ws - 4.00
        ws = img.select("u_component_of_wind_10m").hypot(img.select("v_component_of_wind_10m"))
        hi = t.add(e.multiply(0.33)).subtract(ws.multiply(0.70)).subtract(4.00)
    else:
        hi = heat_index_nws(t, rh)
    return ee.Image(hi.rename("HI").copyProperties(img, ["system:time_start"]))

def _day_key(img):
    return img.set("day", ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"))

def _clip_to(geom):
    def f(img):
        return img.clip(geom)
    return f

def _daily_max(feat):
    hours = ee.ImageCollection.fromImages(feat.get("hours"))
    return hours.max().set("system:time_start", feat.get("system:time_start"))

def exact_median(ic):
    """Mediana EXATA por pixel (ordena o vetor de valores válidos). O ee.Reducer.median() é
    aproximado (histograma): no HEAT de Aracaju errou 0,02 °C."""
    arr = ic.toArray().arrayProject([0]).arraySort()
    n = arr.arrayLength(0)
    ok = n.gt(0)
    arr = arr.updateMask(ok)
    lo = arr.arrayGet(n.subtract(1).divide(2).floor().max(0).toInt())
    hi = arr.arrayGet(n.divide(2).floor().min(n.subtract(1)).max(0).toInt())
    return lo.add(hi).divide(2).updateMask(ok)

def heat_layer(year, ctx):
    """MEDIANA anual (ou p95) do máximo diário do índice de calor na janela da tarde.
    - Resolução ORIGINAL do ERA5-Land (~9 km): sem reprojeção nem interpolação; cada pixel
      de saída recebe o valor da célula ERA5 em que cai (vizinho mais próximo).
    - Máximo diário por JOIN (cada dia junta as suas horas). A versão anterior fazia 365
      filterDate dentro de um map e estourava a memória."""
    h0, h1 = HEAT_LOCAL_HOURS
    u0, u1 = h0 - ctx["utc_offset"], h1 - ctx["utc_offset"]   # local -> UTC
    if not (0 <= u0 <= u1 <= 23):
        raise ValueError(f"Janela UTC inválida {u0}-{u1} para {ctx['name']}")
    box = ctx["region_buf"].bounds(1000)
    if HEAT_FILL_NEIGHBORS:
        box = box.buffer(15000, 1000).bounds(1000)   # + 1 célula ERA5 (~11 km): vizinhas
    era = (ee.ImageCollection(ERA5_HOURLY)
           .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
           .select(["temperature_2m", "dewpoint_temperature_2m",
                    "u_component_of_wind_10m", "v_component_of_wind_10m"]))
    hourly = (era.filter(ee.Filter.calendarRange(u0, u1, "hour"))
              .map(_clip_to(box)).map(_hourly_heat).map(_day_key))
    first = hourly.filter(ee.Filter.calendarRange(u0, u0, "hour"))   # 1 imagem por dia
    joined = ee.Join.saveAll("hours").apply(first, hourly,
                                            ee.Filter.equals(leftField="day", rightField="day"))
    daily = ee.ImageCollection(joined.map(_daily_max))
    if HEAT_STAT == "p95_daily_max":
        heat = daily.reduce(ee.Reducer.percentile([95])).rename("HEAT")
    else:                                     # "median_daily_max" (padrão)
        heat = exact_median(daily).rename("HEAT")   # ee.Reducer.median() é aproximado
    ndays_xc = daily.map(lambda i: i.gte(HEAT_XC_THRESH_C)).sum().rename("HEAT_ndays_xc")
    nat = ee.Image(era.first()).select("temperature_2m").projection()
    heat_flag = ee.Image.constant(0).rename("HEAT_filled")
    if HEAT_FILL_NEIGHBORS:
        # célula sem dado <- média das células vizinhas imediatas com dado (3×3), grade nativa, mesmo ano
        # (skipMasked=False — senão a saída fica mascarada justamente onde a célula central não tem dado)
        K1 = ee.Kernel.square(radius=1, units="pixels")
        h0, n0 = heat.reproject(nat), ndays_xc.reproject(nat)
        hN = h0.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=K1, skipMasked=False).reproject(nat)
        nN = n0.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=K1, skipMasked=False).reproject(nat)
        heat_flag = h0.mask().Not().And(hN.mask()).reproject(nat).rename("HEAT_filled")
        heat, ndays_xc = h0.unmask(hN).rename("HEAT"), n0.unmask(nN).rename("HEAT_ndays_xc")
    if NATIVE_GRID_EVAL or HEAT_FILL_NEIGHBORS:
        # conta feita 1 vez por célula ERA5-Land (0,1°), não por pixel de 30 m: mesmo valor,
        # pois cada pixel de 30 m já recebia o valor da célula em que cai (vizinho mais próximo)
        heat, ndays_xc = heat.reproject(nat), ndays_xc.reproject(nat)
    return heat, ndays_xc, heat_flag


# ============================================================================
# 4) AQ — MAIAC com QA, sem "zero = ar perfeito"
# ============================================================================
def _maiac_prep(img):
    aod = img.select("Optical_Depth_055").multiply(0.001)
    if AQ_REQUIRE_BEST_QA:
        qa = img.select("AOD_QA")
        clear = qa.bitwiseAnd(7).eq(1)                  # bits 0-2 = 001 (céu claro)
        adj   = qa.rightShift(5).bitwiseAnd(7).eq(0)    # bits 5-7 = 000 (sem adjacência)
        best  = qa.rightShift(8).bitwiseAnd(15).eq(0)   # bits 8-11 = 0000 (best quality)
        aod = aod.updateMask(clear.And(adj).And(best))
    return aod.rename("AOD")

def aq_layer(year, ctx):
    """AQ = MEDIANA anual na resolução ORIGINAL (MAIAC 1 km), sem reprojeção. Pixels com
    menos de AOD_MIN_OBS retrievals ficam sem dado; AQ_FILL decide o que fazer com eles."""
    s, e = f"{year}-01-01", f"{year + 1}-01-01"
    if AQ_SOURCE == "pm25":
        if not PM25_COLLECTION:
            raise ValueError("AQ_SOURCE='pm25' exige PM25_COLLECTION e PM25_BAND.")
        aq = ee.ImageCollection(PM25_COLLECTION).filterDate(s, e).select(PM25_BAND).median()
        n = ee.Image.constant(1).rename("n_aod")
    else:
        col = (ee.ImageCollection(MAIAC_IC).filterDate(s, e)
               .filterBounds(ctx["region_buf"]).map(_maiac_prep))
        n = col.count().rename("n_aod")
        aq = col.median().updateMask(n.gte(AOD_MIN_OBS))  # sem dado -> MASCARADO (não 0)
    aq = aq.rename("AQ")
    if AQ_FILL == "city_median":
        # lacuna -> mediana do ano na cidade (um número; nenhuma operação espacial)
        m = aq.reduceRegion(ee.Reducer.median(), ctx["region"], AQ_STATS_SCALE,
                            maxPixels=1e12, bestEffort=True).get("AQ")
        filled = ee.Image(ee.Algorithms.If(m, aq.unmask(ee.Image.constant(m)), aq)).rename("AQ")
    else:
        filled = aq
    flag = aq.mask().Not().And(filled.mask()).rename("AQ_filled")
    return filled, n.unmask(0), flag


# ============================================================================
# 5) Pilha bruta de descritores
# ============================================================================
def raw_stack(year, ctx):
    comp, ls_qa = landsat_composite(year, ctx)
    opt = descriptors_optical(comp)
    heat, heat_nd, heat_flag = heat_layer(year, ctx)
    aq, n_aod, aq_fill = aq_layer(year, ctx)
    raw = ee.Image.cat([opt, heat, aq]).select(DESCRIPTORS)
    qa = ee.Image.cat([ls_qa, heat_nd, heat_flag, n_aod, aq_fill])
    return raw, qa


# ============================================================================
# 6) Normalização (limites agrupados) + EQ (média geométrica ponderada)
# ============================================================================
def normalize_stack(raw, bounds):
    out, flags = [], ee.Image.constant(0)
    for i, name in enumerate(DESCRIPTORS):
        lo, hi = float(bounds[name]["min"]), float(bounds[name]["max"])
        x = raw.select(name)
        if DIRECTION[name] == "benefit":
            ni = x.subtract(lo).divide(hi - lo)
        else:
            ni = ee.Image.constant(hi).subtract(x).divide(hi - lo)
        clipped = ni.lt(NI_FLOOR).Or(ni.gt(1))
        flags = flags.add(clipped.multiply(2 ** i))          # bit i = descritor saturado
        out.append(ni.clamp(NI_FLOOR, 1.0).rename("NI_" + name))
    return ee.Image.cat(out), flags.rename("NI_clamped")

def compute_eq(ni, weights):
    tot = float(sum(weights[d] for d in DESCRIPTORS))
    logsum = ee.Image.constant(0.0)
    for d in DESCRIPTORS:
        logsum = logsum.add(ni.select("NI_" + d).log().multiply(float(weights[d]) / tot))
    return ee.Image(logsum).exp().clamp(NI_FLOOR, 1.0).rename("EQ")


# ============================================================================
# 7) [3] População — GHS-POP R2023A (100 m, resolução ORIGINAL) com interpolação log-linear
# ============================================================================
def _ghs_epoch(e):
    return ee.Image(ee.ImageCollection(GHS_POP_IC)
                    .filter(ee.Filter.calendarRange(e, e, "year")).first()).select(0)

def pop_provenance(year):
    if POP_SOURCE == "worldpop":
        return "observado" if 2000 <= year <= 2020 else "mantido_2020"
    return "observado" if year % 5 == 0 else "interpolado_loglinear"

def pop_year(year):
    """Habitantes por célula de 100 m (sem reprojeção nem agregação)."""
    if POP_SOURCE == "worldpop":
        y = min(max(year, 2000), 2020)
        return ee.Image(ee.ImageCollection(WORLDPOP_IC)
                        .filter(ee.Filter.eq("country", "BRA"))
                        .filter(ee.Filter.eq("year", y)).first()).select("population").rename("POP")
    lo = 5 * (year // 5)
    a = _ghs_epoch(lo)
    if year % 5 == 0:
        return a.rename("POP")
    hi = lo + 5
    b = _ghs_epoch(hi)
    f = (year - lo) / 5.0
    lin = a.add(b.subtract(a).multiply(f))
    loglin = a.max(1e-6).log().multiply(1 - f).add(b.max(1e-6).log().multiply(f)).exp()
    # log-linear onde as duas épocas são > 0; linear onde alguma é 0
    return lin.where(a.gt(0).And(b.gt(0)), loglin).rename("POP")


# ============================================================================
# 8) Vulnerabilidade V e verificação de degenerescência da justiça
# ============================================================================
def _census_img(y):
    spec = V_CENSUS[y]
    fc = ee.FeatureCollection(spec["asset"]).filter(ee.Filter.notNull([spec["prop"]]))
    return fc.reduceToImage([spec["prop"]], ee.Reducer.first())   # valor do setor, sem reamostrar

def vuln_raw(year):
    """Variável socioeconômica (maior = MENOS vulnerável) e proveniência."""
    if V_SOURCE == "census":
        yrs = sorted(int(k) for k in V_CENSUS)
        if not yrs:
            raise ValueError("V_SOURCE='census' exige V_CENSUS preenchido.")
        lo = max([y for y in yrs if y <= year], default=yrs[0])
        hi = min([y for y in yrs if y >= year], default=yrs[-1])
        a = _census_img(lo)
        if lo == hi:
            return a.rename("VAR"), ("observado" if lo == year else "mantido_%d" % lo)
        b = _census_img(hi)
        f = (year - lo) / float(hi - lo)
        x = a.max(1e-6).log().multiply(1 - f).add(b.max(1e-6).log().multiply(f)).exp()
        return x.rename("VAR"), "interpolado_loglinear"
    if not GDP_ASSET:
        raise ValueError("V_SOURCE='gdp_adm1' exige GDP_ASSET na configuração.")
    y = min(max(year, GDP_YEAR_START), GDP_YEAR_END)
    prov = "observado" if y == year else "mantido_%d" % y
    return ee.Image(GDP_ASSET).select(f"PPP_{y}").rename("VAR"), prov

def vulnerability(year, ctx):
    x, prov = vuln_raw(year)
    # percentis da cidade: só uma estatística (escala de amostragem); a camada não muda
    pct = x.reduceRegion(ee.Reducer.percentile([2, 98]), ctx["region"], V_STATS_SCALE,
                         maxPixels=1e12, bestEffort=True)
    p2  = ee.Number(ee.Algorithms.If(pct.get("VAR_p2"),  pct.get("VAR_p2"),  0))
    p98 = ee.Number(ee.Algorithms.If(pct.get("VAR_p98"), pct.get("VAR_p98"), 0))
    rng = p98.subtract(p2)
    # V constante na cidade (ex.: PIB estadual) => J não tem informação
    j_valid = rng.gt(p98.abs().multiply(V_MIN_REL_RANGE).max(1e-12))
    sc = x.subtract(p2).divide(rng.max(1e-12)).clamp(0, 1)
    v = ee.Image.constant(1).subtract(sc).rename("V")
    return v, ee.Number(j_valid), prov


# ============================================================================
# 9) Focais ponderados por população — na grade de SAÍDA (30 m), sem agregação
#    Janela quadrada de WIN_RADIUS_M (em metros). "boxcar" = soma móvel O(1) por pixel.
# ============================================================================
def _kernel():
    return ee.Kernel.square(radius=WIN_RADIUS_M, units="meters")

def _nsum(img, K):
    return img.reduceNeighborhood(reducer=ee.Reducer.sum(), kernel=K, optimization="boxcar")

def _wmean(img, w, K):
    valid = img.mask().gt(0)
    num = _nsum(img.unmask(0).multiply(w).multiply(valid), K)
    # denominador só com pixels onde img é válido
    den = _nsum(w.multiply(valid), K).max(1e-9)
    return num.divide(den)

def equity_local(eq, w):
    K = _kernel()
    mu = _wmean(eq, w, K)
    ex2 = _wmean(eq.pow(2), w, K)
    sd = ex2.subtract(mu.pow(2)).max(0).sqrt()
    cv = sd.divide(mu.max(1e-6)).min(1.0)
    return ee.Image.constant(1).subtract(cv).clamp(NI_FLOOR, 1.0).rename("E_local")

def justice_local(eq, w, v):
    K = _kernel()
    b = ee.Image.constant(1).subtract(eq)                 # ônus ambiental
    v = v.updateMask(eq.mask())
    mb, mv = _wmean(b, w, K), _wmean(v, w, K)
    cov = _wmean(b.multiply(v), w, K).subtract(mb.multiply(mv))
    sb = _wmean(b.pow(2), w, K).subtract(mb.pow(2)).max(0).sqrt()
    sv = _wmean(v.pow(2), w, K).subtract(mv.pow(2)).max(0).sqrt()
    corr = cov.divide(sb.multiply(sv).max(1e-6))          # + = ônus sobre os vulneráveis
    return ee.Image.constant(1).subtract(corr.clamp(0, 1)).clamp(NI_FLOOR, 1.0).rename("J_local")


# ============================================================================
# 10) Fusão  HU-EEQI = EQ^a · E^b · J^c  (expoentes efetivos no servidor)
# ============================================================================
def fuse_index(eq, e_loc, j_loc, j_valid):
    jv = ee.Number(j_valid) if USE_JUSTICE else ee.Number(0)
    c_eff = ee.Number(GAMMA).multiply(jv)
    if J_FALLBACK == "to_eq":
        a_eff = ee.Number(ALPHA).add(ee.Number(GAMMA).subtract(c_eff))
    else:                                   # "renormalize": alfa e beta proporcionais
        a_eff = ee.Number(ALPHA)
    s = a_eff.add(BETA).add(c_eff)
    ex = {"a": a_eff.divide(s), "b": ee.Number(BETA).divide(s), "c": c_eff.divide(s)}
    log = (eq.log().multiply(ex["a"])
           .add(e_loc.log().multiply(ex["b"]))
           .add(j_loc.log().multiply(ex["c"])))
    return ee.Image(log).exp().clamp(0, 1).rename("HU_EEQI"), ex


# ============================================================================
# 11) Montagem do ano/cidade em DUAS passagens — [6] bandas NI_*, n_obs, sensor
#     Nenhuma camada é reprojetada, agregada ou interpolada.
#     1ª passagem (cara, sem calibração): raw_year_image = descritores brutos + QA + POP + V,
#        gravada em RAW_COLLECTION (notebook 00). É a única que lê Landsat/ERA5/MAIAC.
#     Calibração (notebook 01): amostras tiradas dessas imagens já gravadas (rápido).
#     2ª passagem (barata): index_from_raw = normalização, EQ, E, J e HU-EEQI sobre a pilha
#        gravada (notebook 02).
#     build_year_image = as duas passagens em sequência, sem gravar (mesmo resultado; usado nos
#     testes e nas variantes do notebook 05).
# ============================================================================
RAW_QA_BANDS = ["n_obs", "sensor_flag", "n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "LST", "LST_anom",
                "n_aod", "AQ_filled", "HEAT_ndays_xc", "HEAT_filled"]
RAW_BANDS = DESCRIPTORS + RAW_QA_BANDS + ["POP", "V"]

def raw_config():
    """Parâmetros que mudam a pilha bruta (os limites só entram na 2ª passagem)."""
    c = config_snapshot()
    c.pop("BOUNDS_PCT", None)
    c.update({"USE_JUSTICE": USE_JUSTICE, "V_SOURCE": V_SOURCE, "POP_SOURCE": POP_SOURCE,
              "WIN_RADIUS_M": WIN_RADIUS_M})
    return json.loads(json.dumps(c))

def raw_year_image(year, ctx, DOM):
    raw, qa = raw_stack(year, ctx)
    raw = raw.updateMask(DOM)
    pop = pop_year(year)
    if USE_JUSTICE:
        v, j_valid, v_prov = vulnerability(year, ctx)
    else:
        v, j_valid, v_prov = ee.Image.constant(0).rename("V"), ee.Number(0), "n/a"
    lst = qa.select("LST")
    med = lst.updateMask(DOM).reduceRegion(ee.Reducer.median(), ctx["region"], 90,
                                           maxPixels=1e12, bestEffort=True).get("LST")
    lst_anom = lst.subtract(ee.Number(ee.Algorithms.If(med, med, 0))).rename("LST_anom")
    bands = [raw, qa.select(["n_obs", "sensor_flag", "n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "LST"]),
             lst_anom, qa.select(["n_aod", "AQ_filled", "HEAT_ndays_xc", "HEAT_filled"]), pop, v]
    img = ee.Image.cat([ee.Image(b).toFloat() for b in bands]).select(RAW_BANDS)
    # recorte no BUFFER (região + WIN_RADIUS_M + 1 km): a janela de E/J nas bordas do município usa
    # pixels de fora dele, como no cálculo em uma passagem
    return img.clip(ctx["region_buf"]).set({
        "year": year, "city": ctx["name"], "version": VERSION, "stage": "raw", "j_valid": j_valid,
        "prov": json.dumps({"POP": pop_provenance(year), "V": v_prov, "AQ": "observado",
                            "HEAT": "observado"}),
        "raw_config": json.dumps(raw_config()), "export_scale_m": EQ_SCALE,
        "note": ("Pilha bruta (1ª passagem): descritores sem normalização, QA, POP e V. O índice é "
                 "calculado depois (notebook 02) com a calibração tirada destas imagens (notebook 01).")})

def index_from_raw(r, year, ctx, bounds, weights, DOM):
    r = ee.Image(r)
    raw = r.select(DESCRIPTORS)
    ni, clamped = normalize_stack(raw, bounds)
    eq = compute_eq(ni, weights)
    pop, v = r.select("POP"), r.select("V")
    w = pop.unmask(0).add(POP_W_EPS)
    j_valid = ee.Number(r.get("j_valid"))
    e_loc = equity_local(eq, w)
    j_loc = justice_local(eq, w, v) if USE_JUSTICE else ee.Image.constant(1.0).rename("J_local")
    e_loc, j_loc = e_loc.updateMask(eq.mask()), j_loc.updateMask(eq.mask())
    hu, ex = fuse_index(eq, e_loc, j_loc, j_valid)
    bands = [hu, eq, e_loc, j_loc, ni, r.select(["n_obs", "sensor_flag"]), pop, v,
             urban_mask(year), DOM.rename("domain")]
    if EXPORT_DIAGNOSTICS:
        bands += [raw, clamped, r.select(["n_TM", "n_ETM", "n_OLI", "n_obs_thermal", "LST", "LST_anom",
                                          "n_aod", "AQ_filled", "HEAT_ndays_xc", "HEAT_filled"])]
    img = ee.Image.cat([ee.Image(b).toFloat() for b in bands])
    img = img.select(CORE_BANDS + (DIAG_BANDS if EXPORT_DIAGNOSTICS else []))
    props = {
        "year": year, "city": ctx["name"], "version": VERSION,
        "harmonize": str(HARMONIZE), "roy_method": ROY_METHOD, "l7_policy": L7_POLICY,
        "heat_method": HEAT_METHOD, "heat_stat": HEAT_STAT, "aq_source": AQ_SOURCE,
        "aq_fill": AQ_FILL, "pop_source": POP_SOURCE, "v_source": V_SOURCE,
        "j_fallback": J_FALLBACK, "annual_statistic": "median",
        "prov": r.get("prov"),
        "weights": json.dumps(weights), "bounds": json.dumps(bounds),
        "alpha": ALPHA, "beta": BETA, "gamma": GAMMA, "win_radius_m": WIN_RADIUS_M,
        "ni_floor": NI_FLOOR, "export_scale_m": EQ_SCALE,
        "j_valid": j_valid, "exp_eq": ex["a"], "exp_e": ex["b"], "exp_j": ex["c"],
        "note": ("Camadas na resolução original, sem reprojeção/agregação/interpolação: "
                 "Landsat 30 m, ERA5-Land ~9 km, MAIAC 1 km, GHS-POP 100 m, PIB adm1. "
                 "Agregação anual pela mediana. E/J com janela de WIN_RADIUS_M na grade de exportação. "
                 "Índice calculado sobre a pilha bruta gravada (2ª passagem)"),
    }
    return img.clip(ctx["region"]).set(props)

def build_year_image(year, ctx, bounds, weights, DOM):
    return index_from_raw(raw_year_image(year, ctx, DOM), year, ctx, bounds, weights, DOM)

def raw_asset_id_for(ctx, year):
    return f"{RAW_COLLECTION}/huEEQIraw_{ctx['safe']}_{year}"

def raw_export_kwargs(img, ctx, year):
    pyr = {".default": "mean", "sensor_flag": "mode", "AQ_filled": "mode", "HEAT_filled": "mode"}
    return dict(image=img, description=f"huEEQIraw{VERSION}_{ctx['safe']}_{year}"[:100],
                assetId=raw_asset_id_for(ctx, year), region=ctx["region_buf"],
                scale=EQ_SCALE, crs=EXPORT_CRS, maxPixels=1e13, pyramidingPolicy=pyr)

def check_raw_config(ids):
    """Para se alguma pilha bruta gravada foi feita com outra configuração."""
    if not ids:
        return
    cfgs = (ee.ImageCollection.fromImages([ee.Image(i) for i in ids])
            .aggregate_array("raw_config").distinct().getInfo())
    cur, bad = raw_config(), []
    for s in cfgs:
        old = json.loads(s)
        diff = {k: {"bruto": old.get(k), "atual": v} for k, v in cur.items() if old.get(k) != v}
        if diff:
            bad.append(diff)
    if bad:
        raise ValueError(f"Há pilhas brutas em {RAW_COLLECTION} feitas com outra configuração. Apague-as "
                         "(ou volte a configuração) e rode o notebook 00 de novo.\n"
                         + json.dumps(bad, ensure_ascii=False, indent=1))

def asset_id_for(ctx, year):
    return f"{OUT_COLLECTION}/huEEQI_{ctx['safe']}_{year}"

def export_kwargs(img, ctx, year):
    pyr = {".default": "mean", "sensor_flag": "mode", "NI_clamped": "mode", "AQ_filled": "mode",
           "HEAT_filled": "mode", "urban": "mode", "domain": "mode"}
    return dict(image=img, description=f"huEEQIv6_{ctx['safe']}_{year}"[:100],
                assetId=asset_id_for(ctx, year), region=ctx["region"],
                scale=EQ_SCALE, crs=EXPORT_CRS, maxPixels=1e13, pyramidingPolicy=pyr)

def existing_assets(parent):
    ids, token = set(), None
    try:
        while True:
            params = {"parent": parent}
            if token:
                params["pageToken"] = token
            r = ee.data.listAssets(params)
            ids |= {a.get("id") or a.get("name") for a in r.get("assets", [])}
            token = r.get("nextPageToken")
            if not token:
                break
    except Exception as e:
        print("Aviso: não listei assets existentes:", str(e)[:120])
    return ids

def monitor(prefix, max_ops=2000):
    ops = [o for o in ee.data.listOperations()[:max_ops]
           if o.get("metadata", {}).get("description", "").startswith(prefix)]
    states = {}
    for o in ops:
        s = o["metadata"].get("state")
        states[s] = states.get(s, 0) + 1
    print(prefix, states)
    for o in ops:
        if o["metadata"].get("state") == "FAILED":
            print("  FALHOU:", o["metadata"].get("description"), "->",
                  o.get("error", {}).get("message", "")[:200])
    return states


# ============================================================================
# 13) Destinos e [9] preenchimento temporal de lacunas (notebook 02b; usado pelos 04 e 05)
# ============================================================================
def ensure_collection(path, type_="IMAGE_COLLECTION"):
    """Cria a coleção/pasta se não existir. Sem permissão: para AQUI, antes de criar tasks
    (uma task para destino inexistente só falha depois, com "Asset ... not found")."""
    try:
        ee.data.getAsset(path)
        return
    except Exception:
        pass
    try:
        ee.data.createAsset({"type": type_}, path)
        print("Criado:", path)
    except Exception as e:
        raise PermissionError(f"Não consegui criar {path}. A conta autenticada precisa de permissão "
                              "de escrita no projeto/pasta desse caminho. Nenhuma task foi criada.\n"
                              f"Detalhe: {str(e)[:300]}")

def available_bands():
    return CORE_BANDS + (DIAG_BANDS if EXPORT_DIAGNOSTICS else [])

def fill_band_list():
    """Bandas contínuas preenchidas pela cascata (FILL_BANDS presentes no índice)."""
    allb = available_bands()
    return [b for b in FILL_BANDS if b in allb]

def fill_copy_bands():
    """Contagens e marcas copiadas do ano observado (0 = sem observação/sem marca naquele ano)."""
    allb = available_bands()
    return [b for b in FILL_COPY_BANDS if b in allb]

def _masked(bands):
    return ee.Image.constant([0] * len(bands)).rename(bands).toFloat().updateMask(0)

def _value_and_year(img, year, FB):
    """Bandas FB + uma banda de ano por banda, mascarada igual ao valor."""
    v = ee.Image(img).select(FB).toFloat()
    t = (ee.Image.constant([float(year)] * len(FB))
         .rename([b + "__t" for b in FB]).toFloat())
    return v.addBands(t.updateMask(v.mask()))

def _first_valid(series, years, FB):
    """Para cada pixel e banda, o 1º valor válido percorrendo `years` na ordem dada, e o ano dele."""
    if not years:
        return _masked(FB), _masked(FB)
    tb = [b + "__t" for b in FB]
    r = (ee.ImageCollection.fromImages([_value_and_year(series[y], y, FB) for y in years])
         .reduce(ee.Reducer.firstNonNull()).rename(FB + tb))
    return r.select(FB), r.select(tb).rename(FB)

def city_series(col, ctx):
    """{ano: imagem observada} da capital (1 getInfo para saber os anos presentes)."""
    cc = col.filter(ee.Filter.eq("city", ctx["name"])).filter(ee.Filter.inList("year", YEARS))
    years = sorted(set(int(y) for y in cc.aggregate_array("year").getInfo()))
    return {y: ee.Image(cc.filter(ee.Filter.eq("year", y)).first()) for y in years}

def _spatial_mean(img, FB):
    """Tipo 3: média, banda a banda, dos pixels COM valor numa janela quadrada de WIN_RADIUS_M
    (a mesma janela de E e J)."""
    K = _kernel()
    num, den = _nsum(img.unmask(0), K), _nsum(img.mask(), K)
    return num.divide(den.max(1e-9)).updateMask(den.gt(0)).rename(FB)

def fill_year_image(series, year, ctx, DOM):
    """Imagem do `year` sem lacunas no domínio, pela cascata (pixel a pixel, banda a banda), em TODAS
    as bandas contínuas (fill_band_list(): índice, componentes, descritores, LST, POP e V):
      0 observado;
      1 interpolação linear no tempo entre o último ano com dado ANTES e o primeiro DEPOIS
        (lacuna de 1 ano = média dos dois; vale também para o ano inteiro vazio);
      2 ponta da série (só um lado tem dado): repete o ano com dado mais próximo;
      3 sem dado em nenhum ano: média dos pixels com valor no mesmo ano numa janela de WIN_RADIUS_M;
      4 nenhum pixel com valor na janela: mediana da capital no ano (pixels observados; se o ano não
        tiver nenhum, mediana da camada preenchida no tempo).
    fill_type e gap_years descrevem FILL_KEY_BAND; filled_mask marca (bit i) as bandas FB[i] não
    observadas no pixel-ano. series = {ano: ee.Image observada}."""
    FB = fill_band_list()
    present = sorted(series)
    cur = (ee.Image(series[year]).select(FB).toFloat() if year in series else _masked(FB))
    pv, pt = _first_valid(series, [y for y in reversed(present) if y < year], FB)   # antes
    nv, nt = _first_valid(series, [y for y in present if y > year], FB)             # depois
    yr = ee.Image.constant(float(year))
    interp = pv.add(nv.subtract(pv).multiply(yr.subtract(pt).divide(nt.subtract(pt))))   # tipo 1
    edge = pv.unmask(nv)                                                                  # tipo 2
    temporal = cur.unmask(interp).unmask(edge)                                            # tipos 0–2
    spatial = _spatial_mean(temporal.updateMask(DOM), FB)                                 # tipo 3
    tmp = [b + "__tmp" for b in FB]
    med = (cur.addBands(temporal.rename(tmp)).updateMask(DOM)
           .reduceRegion(ee.Reducer.median(), ctx["region"], EQ_SCALE, crs=EXPORT_CRS,
                         maxPixels=1e13, tileScale=4))
    consts = [ee.Image.constant(ee.Number(ee.Algorithms.If(
                  med.get(b), med.get(b), ee.Algorithms.If(med.get(t), med.get(t), -9999))))
              for b, t in zip(FB, tmp)]
    city_med = ee.Image.cat(consts).rename(FB).toFloat()
    city_med = city_med.updateMask(city_med.neq(-9999))
    filled = temporal.unmask(spatial).unmask(city_med)                                    # tipo 4

    k = FILL_KEY_BAND
    ftype = (ee.Image.constant(4).where(spatial.select(k).mask(), 3).where(edge.select(k).mask(), 2)
             .where(interp.select(k).mask(), 1).where(cur.select(k).mask(), 0))
    kpt, knt = pt.select(k), nt.select(k)
    gap = (knt.subtract(kpt)                    # 1: distância entre os dois anos observados
           .unmask(yr.subtract(kpt))            # 2: só há ano antes
           .unmask(knt.subtract(yr))            # 2: só há ano depois
           .unmask(float(len(YEARS)))           # 3 e 4: nunca observado
           .where(cur.select(k).mask(), 0))
    fmask = ee.Image.constant(0)
    for i, b in enumerate(FB):
        fmask = fmask.add(cur.select(b).mask().Not().multiply(2 ** i))
    parts = [filled, ftype.rename("fill_type").toFloat(), gap.rename("gap_years").toFloat(),
             fmask.rename("filled_mask").toFloat()]
    cb = fill_copy_bands()
    if cb:
        parts.append(ee.Image(series[year]).select(cb).unmask(0).toFloat() if year in series
                     else ee.Image.constant([0] * len(cb)).rename(cb).toFloat())
    if "urban" in available_bands():
        parts.append(urban_mask(year).toFloat())
    img = ee.Image.cat(parts).updateMask(DOM).addBands(DOM.rename("domain").toFloat()).clip(ctx["region"])
    props = {"year": year, "city": ctx["name"], "version": VERSION, "source": OUT_COLLECTION,
             "year_observed": int(year in series), "years_present": json.dumps(present),
             "fill_key_band": k, "fill_bands": json.dumps(FB),
             "fill_method": ("0 observado | 1 interpolação linear no tempo entre o último ano com dado antes "
                             "e o primeiro depois | 2 ponta da série: ano com dado mais próximo | 3 nunca "
                             f"observado: média dos pixels com valor no mesmo ano numa janela de {WIN_RADIUS_M} m"
                             " | 4 sem valor na janela: mediana da capital no ano"),
             "note": ("Preenchimento pixel a pixel sobre a série anual exportada (02b). filled_mask: bit i = "
                      "banda fill_bands[i] preenchida. NI_clamped, AQ_filled e HEAT_filled = 0 onde o ano "
                      "não foi observado.")}
    img = img.set(props)
    if year in series:
        img = ee.Image(img.copyProperties(series[year], ["j_valid", "exp_eq", "exp_e", "exp_j", "prov"]))
    return img

def filled_asset_id(ctx, year):
    return f"{FILLED_COLLECTION}/huEEQIf_{ctx['safe']}_{year}"

def fill_export_kwargs(img, ctx, year):
    pyr = {".default": "mean", "fill_type": "mode", "gap_years": "mode", "filled_mask": "mode",
           "sensor_flag": "mode", "NI_clamped": "mode", "AQ_filled": "mode", "HEAT_filled": "mode",
           "urban": "mode", "domain": "mode"}
    return dict(image=img, description=f"huEEQIf{VERSION}_{ctx['safe']}_{year}"[:100],
                assetId=filled_asset_id(ctx, year), region=ctx["region"], scale=EQ_SCALE,
                crs=EXPORT_CRS, maxPixels=1e13, pyramidingPolicy=pyr)

FILL_SUMMARY_COLS = ["n_domain", "n_observed", "n_interp", "n_edge", "n_spatial", "n_city_median",
                     f"n_gap_gt{FILL_REPORT_GAP}", "n_still_missing", "n_any_band_missing"]

def fill_summary_feature(img, ctx):
    """Pixels do domínio por tipo de preenchimento, lacunas > FILL_REPORT_GAP anos e pixels que ainda
    têm alguma banda sem dado (n_still_missing e n_any_band_missing devem ser 0)."""
    ft, gap = img.select("fill_type"), img.select("gap_years")
    still = ft.gte(0).And(img.select(FILL_KEY_BAND).mask().Not())
    any_miss = ft.gte(0).And(img.select(fill_band_list()).mask().reduce(ee.Reducer.min()).eq(0))
    ind = ee.Image.cat([ft.gte(0), ft.eq(0), ft.eq(1), ft.eq(2), ft.eq(3), ft.eq(4),
                        ft.gt(0).And(ft.lt(3)).And(gap.gt(FILL_REPORT_GAP)), still, any_miss]
                       ).rename(FILL_SUMMARY_COLS)
    s = ind.reduceRegion(ee.Reducer.sum().unweighted(), ctx["region"], EQ_SCALE, crs=EXPORT_CRS,
                         maxPixels=1e13, tileScale=4)
    return ee.Feature(None, s).set({"city": ctx["name"], "year": img.get("year")})

def fill_series_np(X, years, groups=None):
    """Referência em numpy da cascata TEMPORAL (linhas = pixels, colunas = anos).
    Devolve (valores preenchidos, fill_type, gap_years). Tipos 0–2 iguais ao GEE. Pixel nunca
    observado: num painel de pontos não há vizinhos espaciais (tipo 3 do GEE), então recebe a
    mediana por ano dos valores OBSERVADOS das linhas do mesmo grupo (cidade) -> tipo 4."""
    import numpy as np, warnings
    X = np.asarray(X, float)
    m, n = X.shape
    t = np.asarray(years, float)
    obs = ~np.isnan(X)
    Xf, typ = X.copy(), np.where(obs, 0.0, 4.0)
    gap = np.where(obs, 0.0, float(len(YEARS)))
    for r in range(m):
        idx = np.nonzero(obs[r])[0]
        if not len(idx):
            continue
        for c in np.nonzero(~obs[r])[0]:
            b, a = idx[idx < c], idx[idx > c]
            if len(b) and len(a):
                b, a = b[-1], a[0]
                Xf[r, c] = X[r, b] + (X[r, a] - X[r, b]) * (t[c] - t[b]) / (t[a] - t[b])
                typ[r, c], gap[r, c] = 1, t[a] - t[b]
            elif len(b):
                Xf[r, c], typ[r, c], gap[r, c] = X[r, b[-1]], 2, t[c] - t[b[-1]]
            else:
                Xf[r, c], typ[r, c], gap[r, c] = X[r, a[0]], 2, t[a[0]] - t[c]
    never = ~obs.any(axis=1)
    if never.any():
        g = (np.zeros(m, int) if groups is None
             else np.unique(np.asarray(groups).astype(str), return_inverse=True)[1])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            for gi in np.unique(g[never]):
                rows = g == gi
                med = np.nanmedian(X[rows], axis=0)                       # observados do ano
                seen = rows & ~never
                if seen.any():
                    med = np.where(np.isnan(med), np.nanmedian(Xf[seen], axis=0), med)
                Xf[rows & never] = med
    return Xf, typ, gap


In [ ]:
# @title Estágio A — amostras de calibração lidas da PILHA BRUTA (notebook 00), 1 task só
import glob
CALIB_YEARS       = YEARS        # todos os anos: os limites cobrem a série inteira
N_POINTS_PER_CITY = 500          # pixels fixos por capital (30 m, no domínio), os mesmos em todos os anos
CALIB_SEED        = 2024         # sorteio dos pixels: capital i usa CALIB_SEED + i
CITY_FILTER       = None         # ex.: ["Aracaju"] para um teste (não use na calibração final)
YEAR_FILTER       = None         # ex.: [2015]
CALIB_RAW_TABLE   = f"calib_{VERSION}_bruto"

DOM  = domain_mask()
CTXS = [c for c in city_contexts() if not CITY_FILTER or c["name"] in CITY_FILTER]
years = YEAR_FILTER or CALIB_YEARS
raw_done = existing_assets(RAW_COLLECTION)
have = [(c, y) for c in CTXS for y in years if raw_asset_id_for(c, y) in raw_done]
missing = [f"{c['name']} {y}" for c in CTXS for y in years if raw_asset_id_for(c, y) not in raw_done]
if missing:
    raise RuntimeError(f"{len(missing)} capital-ano ainda sem pilha bruta em {RAW_COLLECTION}. Rode o notebook 00 "
                       "e espere TODAS as tasks terminarem (os limites precisam da série inteira). "
                       "Nenhuma task foi criada. Ex.: " + ", ".join(missing[:20]))
if CITY_FILTER or YEAR_FILTER:
    print("ATENÇÃO: CITY_FILTER/YEAR_FILTER ativos — a tabela terá só esse recorte (teste).")
check_raw_config([raw_asset_id_for(c, y) for c, y in have])
print(f"{len(CTXS)} cidades × {len(years)} anos com pilha bruta")

def calib_points(ctx):
    """Pixels sorteados UMA vez, só na máscara do domínio (operação barata), com id fixo."""
    pref = ee.String(f"{ctx['idx']}_")
    return (DOM.selfMask().rename("dom")
            .stratifiedSample(numPoints=N_POINTS_PER_CITY, classBand="dom", region=ctx["region"],
                              scale=EQ_SCALE, seed=CALIB_SEED + ctx["idx"], geometries=True, tileScale=4)
            .randomColumn("prand", CALIB_SEED)        # ordem aleatória fixa (equilíbrio no estágio B)
            .map(lambda f: f.set({"pid": pref.cat(f.get("system:index")), "city": ctx["name"]})))

def samples_from(img, y, pts):
    """LÊ a pilha gravada nos pontos (sem Landsat/ERA5/MAIAC). Pixel-ano sem algum descritor sai."""
    return (ee.Image(img).select(DESCRIPTORS + ["n_obs", "sensor_flag", "AQ_filled", "HEAT_filled"])
            .sampleRegions(collection=pts, properties=["pid", "city", "prand"], scale=EQ_SCALE,
                           tileScale=4, geometries=False)
            .map(_setter({"year": y})))

SEL = ["pid", "prand"] + DESCRIPTORS + ["n_obs", "sensor_flag", "AQ_filled", "HEAT_filled", "city", "year"]
if glob.glob(f"{DRIVE_DIR}/{CALIB_RAW_TABLE}*.csv"):
    print(f"{CALIB_RAW_TABLE}.csv já está no Drive: nada a exportar (apague-o para refazer).")
    print("PRÓXIMO PASSO: rode as células 'Estágio B' e 'Estágio C' — o Estágio C grava "
          f"{os.path.basename(CALIB_JSON)}, que o notebook 02 lê.")
else:
    PTS = {c["name"]: calib_points(c) for c in CTXS}
    fc = ee.FeatureCollection([samples_from(ee.Image(raw_asset_id_for(c, y)), y, PTS[c["name"]])
                               for c, y in have]).flatten()
    t = ee.batch.Export.table.toDrive(collection=fc, description=CALIB_RAW_TABLE, folder=DRIVE_FOLDER,
                                      fileNamePrefix=CALIB_RAW_TABLE, fileFormat="CSV", selectors=SEL)
    t.start()
    print(f"1 exportação iniciada ({len(have)} capital-ano) -> Meu Drive/{DRIVE_FOLDER}/{CALIB_RAW_TABLE}.csv")
    print("PRÓXIMO PASSO: quando a task terminar (célula 'Acompanhar') e o CSV aparecer no Drive, rode as "
          f"células 'Estágio B' e 'Estágio C' — o Estágio C grava {os.path.basename(CALIB_JSON)}, que o notebook 02 lê.")


In [ ]:
# @title Acompanhar as exportações (rode até todas ficarem COMPLETED)
monitor("calib_v6_")

In [ ]:
# @title Estágio B — limites, diagnóstico de consistência temporal e pesos
import glob, datetime
import numpy as np, pandas as pd

CALIB_RAW_TABLE = f"calib_{VERSION}_bruto"
files = sorted(glob.glob(f"{DRIVE_DIR}/{CALIB_RAW_TABLE}*.csv"))   # amostras do estágio A (pilha bruta)
assert files, f"{CALIB_RAW_TABLE}.csv ainda não está no Drive (aguarde a task do estágio A)."
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=DESCRIPTORS)   # linhas COMPLETAS
if "pid" in df:
    df = df.drop_duplicates(["pid", "year"])        # arquivo repetido (task rodada 2 vezes)
CALIB_BALANCE_N = 400   # máx. de amostras válidas por capital e ano (peso igual, como na v4); None = todas
if CALIB_BALANCE_N and "prand" in df:
    # as N válidas de menor 'prand' (ordem aleatória fixa dos pontos): mesma escolha no script GEE
    df = (df.sort_values(["city", "year", "prand"]).groupby(["city", "year"]).head(CALIB_BALANCE_N)
            .reset_index(drop=True))
    poucos = df.groupby(["city", "year"]).size()
    poucos = poucos[poucos < CALIB_BALANCE_N]
    if len(poucos):
        print(f"{len(poucos)} capital-anos com menos de {CALIB_BALANCE_N} amostras válidas "
              f"(mín. {poucos.min()}): ex. {poucos.nsmallest(5).to_dict()}")
print(f"{len(files)} arquivos | {len(df)} amostras | {df.city.nunique()} cidades | "
      f"{df.year.nunique()} anos")

def bounds_pct(d, lo_p, hi_p):
    return {k: {"min": float(np.percentile(d[k], lo_p)), "max": float(np.percentile(d[k], hi_p))}
            for k in DESCRIPTORS}

def normalize_np(d, bounds, floor=0.0, ceil=1.0):
    out = {}
    for k in DESCRIPTORS:
        lo, hi = bounds[k]["min"], bounds[k]["max"]
        x = d[k].to_numpy(float)
        ni = (x - lo) / (hi - lo) if DIRECTION[k] == "benefit" else (hi - x) / (hi - lo)
        out[k] = np.clip(ni, floor, ceil)
    return pd.DataFrame(out, index=d.index)

def w_critic(X):
    sigma = X.std(axis=0, ddof=1)
    R = np.corrcoef(X, rowvar=False)
    C = sigma * (1 - R).sum(axis=1)
    return C / C.sum()

def w_ewm(X):
    P = (X + 1e-12) / (X + 1e-12).sum(axis=0, keepdims=True)
    e = -(P * np.log(P)).sum(axis=0) / np.log(len(X))
    d = 1 - e
    return d / d.sum()

def w_pca(X):
    Z = (X - X.mean(0)) / X.std(0, ddof=1)
    vals, vecs = np.linalg.eigh(np.corrcoef(Z, rowvar=False))
    l = vecs[:, np.argmax(vals)]
    l = l if l.sum() >= 0 else -l
    return np.abs(l) / np.abs(l).sum(), l, vals.max() / vals.sum()

def vif(X):
    return np.diag(np.linalg.inv(np.corrcoef(X, rowvar=False)))

BOUND_SETS = {f"p{a}_{b}": bounds_pct(df, a, b) for a, b in [(1, 99), (2, 98), (5, 95)]}
key = f"p{BOUNDS_PCT[0]}_{BOUNDS_PCT[1]}"
if key not in BOUND_SETS:
    BOUND_SETS[key] = bounds_pct(df, *BOUNDS_PCT)
BOUNDS = BOUND_SETS[key]
print("\nLimites", key, json.dumps({k: [round(v["min"], 4), round(v["max"], 4)]
                                    for k, v in BOUNDS.items()}))

# ---- saturação: % fora dos limites, por ano e descritor ----------------------
NIraw = normalize_np(df, BOUNDS, floor=-np.inf, ceil=np.inf)   # sem corte: conta abaixo do piso E acima de 1
sat = (NIraw.lt(NI_FLOOR) | NIraw.gt(1)).groupby(df.year).mean().mul(100).round(1)
print("\n% de pixels saturados (fora de [floor, 1]) por ano:")
print(sat.to_string())

# ---- consistência temporal dos descritores brutos -----------------------------
print("\nMediana agrupada por ano (procure degraus em 2012/2013 e anos atípicos):")
print(df.groupby("year")[DESCRIPTORS].median().round(4).to_string())
df["sensor_group"] = df["sensor_flag"].map(
    lambda v: "+".join(n for n, b in SENSOR_CODE.items() if int(v) & b) if pd.notna(v) else "?")
print("\nMediana por combinação de sensores:")
print(df.groupby("sensor_group")[DESCRIPTORS].median().round(4).to_string())
print("\nObservações limpas por ano (mediana de n_obs):")
print(df.groupby("year")["n_obs"].median().to_string())

# ---- redundância --------------------------------------------------------------
X = normalize_np(df, BOUNDS).to_numpy()
print("\nCorrelação (NI):")
print(pd.DataFrame(np.corrcoef(X, rowvar=False), DESCRIPTORS, DESCRIPTORS).round(2).to_string())
print("VIF:", dict(zip(DESCRIPTORS, np.round(vif(X), 2))))

# ---- pesos --------------------------------------------------------------------
wp, loadings, ev1 = w_pca(X)
WEIGHTS_ALL = {
    "critic": dict(zip(DESCRIPTORS, w_critic(X).round(6).tolist())),
    "ewm":    dict(zip(DESCRIPTORS, w_ewm(X).round(6).tolist())),
    "pca":    dict(zip(DESCRIPTORS, wp.round(6).tolist())),
    "equal":  {k: 1.0 / len(DESCRIPTORS) for k in DESCRIPTORS},
}
print("\nPC1 explica %.1f%% | cargas: %s" % (100 * ev1, dict(zip(DESCRIPTORS, loadings.round(3)))))
bad = [d for d, l in zip(DESCRIPTORS, loadings) if l < 0]
if bad:
    print("  ATENÇÃO: cargas com sinal oposto (descritor não converge com os demais):", bad)

# bootstrap em blocos de cidade (respeita a dependência espacial dentro da cidade)
rng = np.random.default_rng(2024)
cities_ = df.city.unique()
idx_by_city = {c: np.where(df.city.to_numpy() == c)[0] for c in cities_}
boot = []
for _ in range(500):
    pick = rng.choice(cities_, len(cities_), replace=True)
    rows = np.concatenate([idx_by_city[c] for c in pick])
    boot.append(w_critic(X[rows]))
boot = np.array(boot)
CRITIC_CI = {d: [float(np.percentile(boot[:, i], 2.5)), float(np.percentile(boot[:, i], 97.5))]
             for i, d in enumerate(DESCRIPTORS)}

per_year = pd.DataFrame({y: w_critic(X[df.year.to_numpy() == y]) for y in sorted(df.year.unique())},
                        index=DESCRIPTORS).T
print("\nPesos (método -> descritor):")
print(pd.DataFrame(WEIGHTS_ALL).T.round(3).to_string())
print("\nCRITIC IC95% (bootstrap por cidade):", {k: [round(a, 3), round(b, 3)] for k, (a, b) in CRITIC_CI.items()})
print("\nCRITIC por ano — CV entre anos (%):", (100 * per_year.std() / per_year.mean()).round(1).to_dict())

In [ ]:
# @title Estágio C — comparar com a calibração antiga e salvar calibration_v6.json
if os.path.exists(OLD_CALIB_JSON):
    old = json.load(open(OLD_CALIB_JSON))
    print("Pesos antigos (calibration2.json):", {k: round(v, 3) for k, v in old["weights"].items()})
    print("Limites antigos:", {k: [round(v["min"], 3), round(v["max"], 3)] for k, v in old["bounds"].items()})
    print("Obs.: HEAT e AQ mudaram de definição na v5; os limites antigos desses dois não são comparáveis.")

calib = {
    "version": VERSION,
    "created": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "n_samples": int(len(df)), "years": sorted(int(y) for y in df.year.unique()),
    "bounds": BOUNDS, "bounds_by_pct": BOUND_SETS, "bounds_key": key,
    "weights": WEIGHTS_ALL[WEIGHT_METHOD], "weight_method": WEIGHT_METHOD,
    "weights_all": WEIGHTS_ALL, "critic_ci95": CRITIC_CI,
    "critic_by_year": {int(y): row.round(6).to_dict() for y, row in per_year.iterrows()},
    "pca_pc1_explained": float(ev1), "pca_loadings": dict(zip(DESCRIPTORS, loadings.round(6).tolist())),
    "saturation_pct_by_year": {int(y): row.to_dict() for y, row in sat.iterrows()},
    "direction": DIRECTION, "descriptors": DESCRIPTORS,
    "config": config_snapshot(),
}
with open(CALIB_JSON, "w") as f:
    json.dump(calib, f, indent=2, ensure_ascii=False)
print("Salvo em", CALIB_JSON, "| pesos usados (", WEIGHT_METHOD, "):",
      {k: round(v, 3) for k, v in calib["weights"].items()})
assert os.path.exists(CALIB_JSON), f"Não consegui gravar {CALIB_JSON} (confira se o Drive está montado)."
print("PRÓXIMO PASSO: notebook 02 (célula 'Carregar calibração v6' em diante).")
